# Lab 4 · Evaluación y costos

**Sesión Norma+ · Diplomado de Ciencia de Datos e IA para Ciencias Sociales**

Un pipeline que no se mide es una demostración, no un sistema. En este lab respondemos tres preguntas:

1. **¿Qué tan bien extrae criterios?** Precisión, exhaustividad y F1 contra el gold, etapa por etapa.
2. **¿Qué tan bien extrae datos estructurados?** Exactitud por campo, con cada valor respaldado por una cita.
3. **¿Cuánto cuesta?** Lo que gastamos hoy y lo que costaría procesar el acervo completo de la SCJN.

Modelo: `claude-haiku-4-5`. Costo esperado: menos de USD 0.05.

## 0 · Preparación

In [ ]:
!pip install -q anthropic pydantic pdfplumber pandas

In [ ]:
# Descarga del dataset desde el repositorio público del curso (git viene instalado en Colab).
import os, shutil, zipfile
REPO_URL = "https://github.com/benjov/laboratorio-norma-notebooks"

if not os.path.exists("norma_mini_dataset"):
    !git clone -q --depth 1 "$REPO_URL" repo_curso
    if os.path.exists("repo_curso/norma_mini_dataset"):
        shutil.copytree("repo_curso/norma_mini_dataset", "norma_mini_dataset")
    else:
        print("No se pudo descargar de GitHub. Sube norma_mini_dataset.zip a mano:")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("norma_mini_dataset.zip").extractall(".")
print(sorted(os.listdir("norma_mini_dataset")))

In [ ]:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [ ]:
# Guardián de presupuesto: envuelve al cliente, suma el costo de cada llamada y se detiene al llegar al límite.
# Precios en USD por millón de tokens (entrada, salida). Un modelo no listado usa un precio conservador.
PRECIOS = {"claude-haiku-4-5": (1.0, 5.0)}
PRECIO_CONSERVADOR = (5.0, 25.0)

class Presupuesto:
    def __init__(self, client, limite_usd):
        self.client, self.limite, self.gastado, self.llamadas = client, limite_usd, 0.0, 0
        self.registro = []  # una fila por llamada: modelo, tokens y costo

    def crear(self, **kwargs):
        if self.gastado >= self.limite:
            raise RuntimeError(f"Límite de USD {self.limite:.2f} alcanzado en este notebook.")
        r = self.client.messages.create(**kwargs)
        p_in, p_out = PRECIOS.get(kwargs["model"], PRECIO_CONSERVADOR)
        costo = (r.usage.input_tokens * p_in + r.usage.output_tokens * p_out) / 1e6
        self.gastado += costo
        self.llamadas += 1
        self.registro.append({"modelo": kwargs["model"], "entrada": r.usage.input_tokens,
                              "salida": r.usage.output_tokens, "costo": costo})
        return r

    def __repr__(self):
        return f"{self.llamadas} llamadas · USD {self.gastado:.4f} de {self.limite:.2f}"

In [ ]:
import re, json, hashlib, difflib, unicodedata
from pathlib import Path
from typing import Literal
import pandas as pd
import pdfplumber
from pydantic import BaseModel, Field, ValidationError

DATA = Path("norma_mini_dataset")
SALIDAS = DATA / "salidas"
SALIDAS.mkdir(exist_ok=True)
MODELO = "claude-haiku-4-5"
api = Presupuesto(client, limite_usd=0.50)

gold = pd.read_csv(DATA / "gold_criterios.csv")
gold_info = pd.read_csv(DATA / "gold_info_general.csv", dtype=str).fillna("")

def normalizar(s):
    return re.sub(r"\s+", " ", str(s)).strip()

## 1 · Criterios: precisión, exhaustividad y F1

Emparejamos cada extracción con **a lo más un** criterio del gold del mismo documento, cuando sus citas se traslapan. Así nadie cuenta doble. Después contamos:

- **VP:** extracciones que corresponden a un criterio del gold;
- **FP:** extracciones que no corresponden a ninguno;
- **FN:** criterios del gold que nadie encontró.

### ✏️ Ejercicio 1
Completa `prf`: devuelve precisión, exhaustividad y F1 (la media armónica de ambas). Cuida las divisiones entre cero.

In [ ]:
def prf(vp, fp, fn):
    p = vp / (vp + fp) if vp + fp else 0.0
    r = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 3), round(r, 3), round(f1, 3)

In [ ]:
assert prf(8, 2, 2) == (0.8, 0.8, 0.8) and prf(0, 0, 5) == (0.0, 0.0, 0.0)
print("✅ prf correcta")

def similitud(a, b):
    a, b = normalizar(a).lower(), normalizar(b).lower()
    if a in b or b in a:
        return 1.0
    return difflib.SequenceMatcher(None, a.split(), b.split()).ratio()

def emparejar(extraidos, gold, umbral=0.6):
    """Emparejamiento uno a uno, de la pareja más parecida a la menos, dentro de cada documento."""
    parejas = sorted(((similitud(e.cita_literal, g.cita_literal), i, j)
                      for i, e in extraidos.iterrows() for j, g in gold.iterrows()
                      if e.documento == g.documento), reverse=True)
    usados_e, usados_g, match = set(), set(), {}
    for s, i, j in parejas:
        if s >= umbral and i not in usados_e and j not in usados_g:
            usados_e.add(i); usados_g.add(j); match[i] = j
    return match

def evaluar(extraidos, nombre):
    m = emparejar(extraidos, gold)
    vp, fp, fn = len(m), len(extraidos) - len(m), len(gold) - len(m)
    p, r, f1 = prf(vp, fp, fn)
    tema = sum(extraidos.loc[i].id_tema == gold.loc[j].id_tema for i, j in m.items()) / max(len(m), 1)
    return {"etapa": nombre, "n": len(extraidos), "VP": vp, "FP": fp, "FN": fn,
            "precision": p, "exhaustividad": r, "F1": f1, "tema_correcto": round(tema, 3)}

In [ ]:
etapas = []
r2, rv = SALIDAS / "criterios_lab2.jsonl", SALIDAS / "criterios_validados.jsonl"
if r2.exists():
    lab2 = pd.read_json(r2, lines=True)
    lab2 = lab2[lab2.estado != "esquema_invalido"].reset_index(drop=True)
    etapas.append(evaluar(lab2, "1 · Extracción (todo)"))
    etapas.append(evaluar(lab2[lab2.estado == "verificado"].reset_index(drop=True), "2 · Cita verificada"))
if rv.exists():
    val = pd.read_json(rv, lines=True)
    if "estado" in val:  # solo si viene del Lab 2, no del respaldo con el gold
        etapas.append(evaluar(val[val.aceptado].reset_index(drop=True), "3 · Validador + gate"))

if etapas:
    tabla = pd.DataFrame(etapas)
    display(tabla)
else:
    print("No hay extracciones del Lab 2 en salidas/. Corre el Lab 2 y el bloque de validación para evaluar tu pipeline.")

> **Para discutir:** compara las filas. Cada control debería **subir la precisión**. ¿Cuánto **bajó la exhaustividad** a cambio? Ese intercambio es una decisión de producto: una base para litigio prioriza precisión; una exploración académica quizá prefiera exhaustividad. Mira también `tema_correcto`: una extracción puede ser correcta y estar mal clasificada, y entonces nadie la encuentra al navegar la taxonomía.

## 2 · Información general: exactitud por campo

Además de criterios, Norma+ extrae datos estructurados de cada resolución: expediente, fechas, agentes, sentido, montos, votación. Aquí aplicamos una regla estricta, **prueba de cierre**: cada valor debe venir con una cita literal del documento. Si la cita no está en el texto, el valor se descarta, aunque parezca correcto.

`sentido` y `votacion` son categorías cerradas. Comparar texto libre («Se acredita la responsabilidad» frente a «Responsabilidad acreditada») haría que la métrica mida redacción, no exactitud.

In [ ]:
def texto_pdf(ruta):
    with pdfplumber.open(ruta) as pdf:
        lineas = [[l["text"] for l in p.extract_text_lines()] for p in pdf.pages]
    firma = lambda t: re.sub(r"\d+", "#", t.strip())
    ultimas = [firma(ls[-1]) for ls in lineas if ls]
    pie = {f for f in ultimas if ultimas.count(f) >= 2}
    return normalizar(" ".join(l for ls in lineas for l in ls if firma(l) not in pie))

TEXTOS = {p.name: texto_pdf(p) for p in sorted((DATA / "pdf").glob("*.pdf"))}
PAGINAS = {p.name: len(pdfplumber.open(p).pages) for p in sorted((DATA / "pdf").glob("*.pdf"))}

class Campo(BaseModel):
    valor: str = Field(description="El valor normalizado; cadena vacía si el documento no lo contiene.")
    cita: str = Field(description="Fragmento copiado EXACTAMENTE del documento que respalda el valor; vacío si no aplica.")

class Sentido(BaseModel):
    valor: Literal[("sancion", "absolucion", "autorizacion", "autorizacion_con_condiciones", "objecion",
                    "ampara", "ampara_en_parte", "niega_amparo", "sobresee")] = Field(description="Sentido de la resolución.")
    cita: str = Field(description="Fragmento literal que lo respalda.")

class Votacion(BaseModel):
    valor: Literal[("unanimidad", "mayoria", "no_aplica")] = Field(description="no_aplica si resuelve un órgano unipersonal.")
    cita: str = Field(description="Fragmento literal que lo respalda; vacío si no aplica.")

class InfoGeneral(BaseModel):
    expediente: Campo = Field(description="Número de expediente o de juicio, tal como aparece.")
    fecha_resolucion: Campo = Field(description="Fecha de la resolución o sentencia, en formato AAAA-MM-DD.")
    fecha_demanda: Campo = Field(description="Fecha de presentación de la demanda, AAAA-MM-DD; vacío si no es un juicio.")
    agentes: Campo = Field(description="Agentes económicos investigados, notificantes o quejosos, separados por '; '.")
    sentido: Sentido
    multa_mxn: Campo = Field(description="Multa impuesta en pesos, como número entero sin separadores; vacío si no hay.")
    monto_operacion_mxn: Campo = Field(description="Monto de la operación en pesos, número entero; vacío si no hay.")
    votacion: Votacion

def esquema_plano(modelo):
    s = modelo.model_json_schema()
    defs = s.pop("$defs", {})
    def resolver(x):
        if isinstance(x, dict):
            if "$ref" in x:
                return resolver(defs[x["$ref"].split("/")[-1]])
            return {k: resolver(v) for k, v in x.items()}
        return [resolver(v) for v in x] if isinstance(x, list) else x
    return resolver(s)

HERR_INFO = {"name": "registrar_info", "description": "Registra los datos generales de la resolución.",
             "input_schema": esquema_plano(InfoGeneral)}
SISTEMA_INFO = """Extrae los datos generales de una resolución mexicana de competencia económica.
Cada valor debe ir acompañado de la cita literal que lo respalda, copiada exactamente del documento.
Si un dato no aparece, deja valor y cita vacíos. Nunca infieras un dato que el texto no diga."""

CACHE_RUTA = SALIDAS / "cache_lab4.json"
cache = json.load(open(CACHE_RUTA, encoding="utf-8")) if CACHE_RUTA.exists() else {}

def extraer_info(documento):
    usuario = f"DOCUMENTO:\n{TEXTOS[documento]}"
    clave = hashlib.sha256((MODELO + SISTEMA_INFO + usuario).encode()).hexdigest()
    if clave not in cache:
        r = api.crear(model=MODELO, max_tokens=1200, system=SISTEMA_INFO, tools=[HERR_INFO],
                      tool_choice={"type": "tool", "name": "registrar_info"},
                      messages=[{"role": "user", "content": usuario}])
        cache[clave] = next(b.input for b in r.content if b.type == "tool_use")
        json.dump(cache, open(CACHE_RUTA, "w", encoding="utf-8"), ensure_ascii=False)
    try:
        return InfoGeneral.model_validate(cache[clave])
    except ValidationError:
        return None

info = {d: extraer_info(d) for d in TEXTOS}
print(api)

In [ ]:
CAMPOS = ["expediente", "fecha_resolucion", "fecha_demanda", "agentes", "sentido", "multa_mxn", "monto_operacion_mxn", "votacion"]

def con_cierre(documento, campo):
    """Prueba de cierre: el valor solo cuenta si su cita existe literalmente en el documento."""
    ig = info[documento]
    if ig is None:
        return ""
    c = getattr(ig, campo)
    if c.valor in ("", "no_aplica"):
        return c.valor
    return c.valor if c.cita and normalizar(c.cita) in TEXTOS[documento] else "⛔ sin respaldo"

### ✏️ Ejercicio 2 · Comparar como lo haría una persona

Un modelo puede escribir `"$38,450,000.00"` y el gold `"38450000"`. Completa `normalizar_monto` para que ambos queden como el entero `38450000`: quita todo lo que no sea dígito antes del punto decimal. Una cadena vacía debe devolver `None`.

In [ ]:
def normalizar_monto(s):
    s = str(s).split(".")[0]
    digitos = re.sub(r"\D", "", s)
    return int(digitos) if digitos else None

In [ ]:
assert normalizar_monto("$38,450,000.00") == 38450000 == normalizar_monto("38450000")
assert normalizar_monto("") is None
print("✅ normalizar_monto correcta")

SUFIJOS = {"s", "a", "de", "c", "v", "p", "i", "sa", "cv", "sapi"}
def tokens_agentes(s):
    s = unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
    return {t for t in re.findall(r"[a-z0-9]+", s) if t not in SUFIJOS}

def correcto(campo, pred, real):
    if pred == "⛔ sin respaldo":
        return False
    if campo in ("multa_mxn", "monto_operacion_mxn"):
        return normalizar_monto(pred) == normalizar_monto(real)
    if campo == "agentes":
        a, b = tokens_agentes(pred), tokens_agentes(real)
        return (not a and not b) or (len(a & b) / max(len(a | b), 1) >= 0.6)
    if campo == "expediente":
        return normalizar(pred).upper() == normalizar(real).upper()
    return normalizar(pred) == normalizar(real)

filas = []
for _, g in gold_info.iterrows():
    for campo in CAMPOS:
        pred = con_cierre(g.documento, campo)
        filas.append({"documento": g.documento[:2], "campo": campo, "predicho": pred,
                      "gold": g[campo], "correcto": correcto(campo, pred, g[campo])})
eval_info = pd.DataFrame(filas)
print(f"Exactitud global: {eval_info.correcto.mean():.2f} ({eval_info.correcto.sum()} de {len(eval_info)})\n")
print(eval_info.groupby("campo").correcto.mean().round(2).to_string())
eval_info[~eval_info.correcto]

> **Para discutir:** ¿los errores son del modelo o de la regla de comparación? Un `⛔ sin respaldo` puede ser un valor correcto con una cita mal copiada. La prueba de cierre lo descarta igual: preferimos un hueco a un dato que nadie puede verificar.

## 3 · Costos: de hoy al acervo de la SCJN

El guardián registró cada llamada de este notebook. Con los tokens que **medimos** al leer documentos completos, proyectamos el costo de procesar un corpus grande.

El deck de Norma+ estima unas 315 mil resoluciones de la SCJN desde 1995, con unos 63 millones de páginas.

In [ ]:
reg = pd.DataFrame(api.registro)
print(api)
if len(reg):
    paginas = sum(PAGINAS.values())
    tokens_pagina = reg.entrada.sum() / paginas  # incluye el prompt de sistema y el esquema: un estimador conservador
    salida_doc = reg.salida.mean()
    print(f"Medido: {tokens_pagina:,.0f} tokens de entrada por página · {salida_doc:,.0f} tokens de salida por documento")

### ✏️ Ejercicio 3 · ¿Cuánto costaría el corpus?

Completa `costo_corpus`. Con *prompt caching*, la parte repetida del prompt (instrucciones y esquema) se cobra a una fracción del precio en las llamadas siguientes; según la documentación de precios de Anthropic, las lecturas de caché cuestan alrededor del 10% del precio normal de entrada. El parámetro `fraccion_cacheada` indica qué parte de los tokens de entrada se lee de caché.

In [ ]:
def costo_corpus(paginas, tokens_pagina, documentos, salida_doc, precio_in, precio_out, fraccion_cacheada=0.0):
    entrada = paginas * tokens_pagina
    costo_in = entrada * (1 - fraccion_cacheada) * precio_in + entrada * fraccion_cacheada * precio_in * 0.10
    costo_out = documentos * salida_doc * precio_out
    return (costo_in + costo_out) / 1e6

In [ ]:
assert abs(costo_corpus(1_000_000, 1, 0, 0, 1.0, 5.0) - 1.0) < 1e-9
assert abs(costo_corpus(1_000_000, 1, 0, 0, 1.0, 5.0, fraccion_cacheada=1.0) - 0.1) < 1e-9
print("✅ costo_corpus correcta")

if len(reg):
    escenarios = []
    for nombre, (p_in, p_out) in {"Haiku 4.5": PRECIOS["claude-haiku-4-5"]}.items():
        for cache_frac in (0.0, 0.3):
            usd = costo_corpus(63_000_000, tokens_pagina, 315_000, salida_doc, p_in, p_out, cache_frac)
            escenarios.append({"modelo": nombre, "fracción en caché": cache_frac,
                               "USD solo info general": round(usd), "USD por resolución": round(usd / 315_000, 4)})
    display(pd.DataFrame(escenarios))

> **Para discutir:** esta proyección cubre **una sola pasada** de información general. El pipeline de criterios hace varias llamadas por chunk (extracción, validación, gate, juez), así que multiplica. ¿Qué etapas conviene hacer con un modelo pequeño y cuáles justifican uno grande? ¿Qué parte se puede resolver sin LLM, como hicimos en el Lab 1? Así se piensa la ingeniería de costos en Norma+: medir tokens por etapa, cachear prefijos y reservar el modelo caro para lo que lo necesita.

## Cierre de la sesión

| Lab | Técnica | Lección |
|---|---|---|
| 1 | Estructura con reglas | Lo que se puede resolver sin LLM, se resuelve sin LLM |
| 2 | Extracción con esquema | El esquema y la verificación literal valen más que el prompt |
| 2b | Validador y gate | Cada control se mide antes de confiar en él |
| 3 | Dedup en capas | Proponer barato, decidir caro, vetar con reglas auditables |
| 4 | Evaluación y costos | Sin gold no hay sistema; sin medir tokens no hay negocio |

**Límites y sesgos para llevarse:**
- El gold lo escribieron personas con un criterio propio: medimos acuerdo con *ellas*, no la verdad.
- Nuestros documentos son cortos y limpios; las resoluciones reales tienen cientos de páginas, escaneos y tablas.
- Un modelo que falla de forma sistemática en un tipo de procedimiento sesga la base completa: hay que evaluar por estrato, no solo en promedio.

**Para ciencias sociales:** este flujo es análisis de contenido a escala con trazabilidad. Cada dato apunta a su fuente; cada etapa tiene su métrica; cada decisión automática puede auditarse.